# Use Case 1 — Chat Completions

This notebook walks through three patterns for the OpenAI chat completions API:

1. **Basic** — single Q&A with an optional system prompt
2. **Streaming** — print tokens as they arrive
3. **Structured output** — parse the response into a Pydantic model

**Prerequisites:** Set `OPENAI_API_KEY` in the kernel environment or a `.env` file.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env if present

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError("OPENAI_API_KEY is not set")

MODEL = os.environ.get("CHAT_MODEL", "gpt-4o")
print(f"Using model: {MODEL}")

## 1. Basic Chat

`client.chat.completions.create()` — synchronous, returns a full `ChatCompletion` object.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

completion = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"},
    ],
)

response = completion.choices[0].message.content
print("Response:", response)
print("Tokens used:", completion.usage.total_tokens)

### Multi-turn with history

Pass prior messages in the `messages` list to continue a conversation.

In [ ]:
history = [
    {"role": "user", "content": "My name is Alice."},
    {"role": "assistant", "content": "Nice to meet you, Alice!"},
]

completion = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        *history,
        {"role": "user", "content": "What is my name?"},
    ],
)

print(completion.choices[0].message.content)

## 2. Streaming

`client.chat.completions.stream()` — context manager that yields `StreamEvent` objects.
Listen for `content.delta` events to print tokens as they arrive.

In [ ]:
print("Streaming response:\n")

with client.chat.completions.stream(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a two-sentence description of Python."},
    ],
) as stream:
    for event in stream:
        if event.type == "content.delta" and event.content:
            print(event.content, end="", flush=True)

print()  # final newline

## 3. Structured Output

`client.beta.chat.completions.parse()` — pass a Pydantic model as `response_format`.
The SDK converts it to a JSON schema, sends it to the model, then parses the response back.

In [ ]:
from pydantic import BaseModel


class ExtractedInfo(BaseModel):
    name: str
    age: int
    summary: str


text = "Bob is a 28-year-old software engineer who loves hiking and open-source projects."

completion = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "Extract the person's name, age, and write a one-sentence summary.",
        },
        {"role": "user", "content": text},
    ],
    response_format=ExtractedInfo,
)

message = completion.choices[0].message
if message.refusal:
    print("Model refused:", message.refusal)
else:
    info = message.parsed
    print(f"Name   : {info.name}")
    print(f"Age    : {info.age}")
    print(f"Summary: {info.summary}")